In [2]:
from Bio import motifs, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import numpy as np

Цель: научиться работать с реальными геномными данными и модулем Bio.motifs.

Задачи:
1. Используя модуль Bio.motifs из Biopython, создайте мотив из последовательностей Задания 1.
2. Загрузите файл chr1.fa (первая хромосома человека) с помощью SeqIO.parse.
3. Извлеките первые 1 000 000 нуклеотидов для анализа.
4. Используйте метод m.pwm.search(sequence, threshold=...) для поиска сайтов связывания.
5. Найдите все хиты (позицию и скор) со скором больше 5.0.
6. Повторите поиск для обратной комплементарной цепи.

Подсказка: Транскрипционные факторы могут связываться на обеих цепях ДНК!

Ответ: Код поиска по хромосоме. Список найденных хитов (позиция, цепь, скор), превышающих порог
5.0, сохраненный в текстовый файл или выведенный в ноутбуке.

In [34]:
#1
sites = [ " GAGGTAAAC " , " TCCGTAAGC " , " CAGGTTGGA " ,
" ACAGTCAGC " , " TAGGTCAGC " , " CAGGTCAGC " ,
" CAGGTCGAT " , " CAGGTCAGC " , " CAGGTCAGC " ,
" CAGGTTGGC " ]
sequences = []
for seq in sites:
    sequences.append(Seq(seq))
print(sequences)
instances = [SeqRecord(s) for s in sequences]
m = motifs.create(instances)
background = {'A': 0.295, 'C': 0.205, 'G': 0.205, 'T': 0.295}
m = motifs.create(sequences)
pssm = m.pssm
print(len(m))


[Seq(' GAGGTAAAC '), Seq(' TCCGTAAGC '), Seq(' CAGGTTGGA '), Seq(' ACAGTCAGC '), Seq(' TAGGTCAGC '), Seq(' CAGGTCAGC '), Seq(' CAGGTCGAT '), Seq(' CAGGTCAGC '), Seq(' CAGGTCAGC '), Seq(' CAGGTTGGC ')]
11


In [15]:
m

In [22]:
#2
fasta_file = "chr1.fa" 
records = list(SeqIO.parse(fasta_file, "fasta"))
chr1_seq = str(records[0].seq)

In [30]:
#3
analysis_seq = chr1_seq[:1000000]
pwm = m.counts.normalize(pseudocounts=0.1)
print(pwm)

        0      1      2      3      4      5      6      7      8      9     10
A:   0.25   0.11   0.78   0.11   0.01   0.01   0.20   0.68   0.20   0.11   0.25
C:   0.25   0.59   0.20   0.11   0.01   0.01   0.59   0.01   0.01   0.78   0.25
G:   0.25   0.11   0.01   0.78   0.97   0.01   0.01   0.30   0.78   0.01   0.25
T:   0.25   0.20   0.01   0.01   0.01   0.97   0.20   0.01   0.01   0.11   0.25



In [37]:
#4 и 5
hits = []
threshold = 5.0

for pos, score in pssm.search(chr1_seq, threshold=threshold):
    hits.append((pos, '+', score))

In [38]:
#6
m_rc = m.reverse_complement()
pssm_rc = m_rc.pssm

for pos, score in pssm_rc.search(chr1_seq, threshold=threshold):
    hits.append((pos, '-', score))

hits.sort(key=lambda x: x[2], reverse=True)

print(f"всего хитов (> {threshold}): {len(hits)}")

output_filename = "binding_sites_chr1.txt"
with open(output_filename, "w") as f:
    f.write("Position\tStrand\tScore\n")
    for pos, strand, score in hits:
        f.write(f"{pos}\t{strand}\t{score:.3f}\n")


Всего найдено хитов (> 5.0): 0

Топ-15 хитов:

Все хиты сохранены в файл: binding_sites_chr1.txt


In [39]:
#вроде все норм, правда 0 хитов